In [ ]:
import os
import math
import time
import copy
import zipfile
import random
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


# ============================================================
# 0. Matplotlib / Font Settings
# ============================================================
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "serif",
    "font.serif": ["Liberation Serif", "FreeSerif", "serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "mathtext.fontset": "stix",
})


# ============================================================
# 1. Unified Configuration
# ============================================================
CFG: Dict = {
    "seed": 0,
    "seeds": [0, 1, 2, 3, 4],

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "dtype": torch.float32,

    # dynamics / cost
    "a0": -0.5,
    "b0": 1.0,
    "b1": 0.8,
    "sigma": 0.2,
    "R": 1.0,
    "S0": -10.0,

    # time grid
    "T": 2.0,
    "N_steps": 20,
    "delta": 1.0,

    # initial condition
    "x0": 0.0,

    # Algorithm 1: warm-up / LSTM-DPO
    "batch_size": 256,
    "warmup_iters": 10000,
    "lr": 1e-4,

    # Algorithm 2: MV-FABSDE
    "fbsde_iters": 10000,
    "fbsde_lr": 5e-4,
    "fbsde_decay_step": 10000,

    # Stage-II projection: P-PGDPO
    "N_mc_stage2": 8,
    "N_branch": 10,
    "N_compare": 51,

    # network
    "hidden": 64,

    # output
    "outdir": "benchmark2_multiseed_figures",
}

DEVICE = torch.device(CFG["device"])
torch.set_default_dtype(CFG["dtype"])
os.makedirs(CFG["outdir"], exist_ok=True)


# ============================================================
# 2. Utilities
# ============================================================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cfg_for_seed(cfg: Dict, seed: int) -> Dict:
    out = copy.deepcopy(cfg)
    out["seed"] = int(seed)
    return out


def dt(cfg: Dict) -> float:
    return cfg["T"] / cfg["N_steps"]


def delay_steps(cfg: Dict) -> int:
    d = int(round(cfg["delta"] / dt(cfg)))
    return max(1, d)


def make_time_norm(n: int, cfg: Dict, batch: int, device: torch.device) -> torch.Tensor:
    t = n * dt(cfg)
    return torch.full((batch,), t / cfg["T"], device=device)


def drift_x(x: torch.Tensor, v: torch.Tensor, v_delayed: torch.Tensor, cfg: Dict) -> torch.Tensor:
    return cfg["a0"] * x + cfg["b0"] * v + cfg["b1"] * v_delayed


def get_v_delayed(v_hist: torch.Tensor, n: int, D: int, device: torch.device) -> torch.Tensor:
    if n - D >= 0:
        return v_hist[:, n - D]
    return torch.zeros(v_hist.size(0), device=device, dtype=v_hist.dtype)


# ============================================================
# 3. Models
# ============================================================
class LSTMPolicy(nn.Module):
    def __init__(self, input_size: int = 2, hidden_size: int = 64):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTMCell(input_size, hidden_size)
        self.head = nn.Linear(hidden_size, 1)
        with torch.no_grad():
            self.head.bias.fill_(0.5)

    def init_hidden(self, batch_size: int, device: torch.device):
        h0 = torch.zeros(batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(batch_size, self.hidden_size, device=device)
        return h0, c0

    def forward_step(self, t_norm, x_t, h, c):
        inp = torch.stack([t_norm, x_t], dim=-1)
        h_next, c_next = self.lstm(inp, (h, c))
        v_raw = self.head(h_next).squeeze(-1)
        v = torch.nn.functional.softplus(v_raw)
        return v, h_next, c_next


class FBSDENet(nn.Module):
    def __init__(self, input_size: int = 2, hidden_size: int = 64):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTMCell(input_size, hidden_size)
        self.head_Y = nn.Linear(hidden_size, 1)
        self.head_Z = nn.Linear(hidden_size, 1)
        self.head_EY = nn.Linear(hidden_size, 1)
        with torch.no_grad():
            self.head_Y.bias.fill_(-5.0)

    def init_hidden(self, batch_size: int, device: torch.device):
        h0 = torch.zeros(batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(batch_size, self.hidden_size, device=device)
        return h0, c0

    def forward_step(self, t_norm, x_t, h, c):
        inp = torch.stack([t_norm, x_t], dim=-1)
        h_next, c_next = self.lstm(inp, (h, c))
        y = self.head_Y(h_next).squeeze(-1)
        z = self.head_Z(h_next).squeeze(-1)
        ey = self.head_EY(h_next).squeeze(-1)
        return y, z, ey, h_next, c_next


# ============================================================
# 4. Algorithm 1: Warm-up / LSTM-DPO
# ============================================================
def simulate_batch_episode_pg(policy: LSTMPolicy, cfg: Dict, detach_policy: bool = False) -> torch.Tensor:
    N, D = cfg["N_steps"], delay_steps(cfg)
    batch = cfg["batch_size"]

    x = torch.full((batch,), cfg["x0"], device=DEVICE)
    v_hist = torch.zeros(batch, N + 1, device=DEVICE)

    h, c = policy.init_hidden(batch, DEVICE)
    cost = torch.zeros(batch, device=DEVICE)

    for n in range(N):
        t_norm = make_time_norm(n, cfg, batch, DEVICE)

        if detach_policy:
            with torch.no_grad():
                v, h, c = policy.forward_step(t_norm, x, h, c)
        else:
            v, h, c = policy.forward_step(t_norm, x, h, c)

        v_hist[:, n] = v
        v_del = get_v_delayed(v_hist, n, D, DEVICE)

        dB = math.sqrt(dt(cfg)) * torch.randn(batch, device=DEVICE)
        x = x + drift_x(x, v, v_del, cfg) * dt(cfg) + cfg["sigma"] * dB
        cost = cost + cfg["R"] * v**2 * dt(cfg)

    cost = cost + cfg["S0"] * x
    return cost.mean()


def warmup_train(policy: LSTMPolicy, cfg: Dict) -> LSTMPolicy:
    policy.to(DEVICE)
    policy.train()
    opt = optim.Adam(policy.parameters(), lr=cfg["lr"])

    for it in range(cfg["warmup_iters"]):
        opt.zero_grad(set_to_none=True)
        J = simulate_batch_episode_pg(policy, cfg, detach_policy=False)
        J.backward()
        opt.step()

        if (it + 1) % 500 == 0:
            print(f"[Warmup] iter={it+1}, J={J.item():.4f}")

    return policy


def rollout_policy_pg_deterministic(policy: LSTMPolicy, cfg: Dict) -> Dict[str, List[torch.Tensor]]:
    N, D = cfg["N_steps"], delay_steps(cfg)
    policy.eval()

    x = torch.full((1,), cfg["x0"], device=DEVICE)
    v_hist = torch.zeros(1, N + 1, device=DEVICE)
    h, c = policy.init_hidden(1, DEVICE)

    xs, vs, hs, cs, v_hists = [], [], [], [], []

    with torch.no_grad():
        for n in range(N):
            xs.append(x.clone())
            hs.append(h.clone())
            cs.append(c.clone())
            v_hists.append(v_hist.clone())

            t_norm = torch.full((1,), (n * dt(cfg)) / cfg["T"], device=DEVICE)
            v, h, c = policy.forward_step(t_norm, x, h, c)
            vs.append(v.clone())

            v_hist[:, n] = v
            v_del = get_v_delayed(v_hist, n, D, DEVICE)
            x = x + drift_x(x, v, v_del, cfg) * dt(cfg)

        xs.append(x.clone())

    return {"xs": xs, "vs": vs, "hs": hs, "cs": cs, "v_hists": v_hists}


# ============================================================
# 5. Algorithm 2: MV-FABSDE
# ============================================================
def train_fbsde(net: FBSDENet, cfg: Dict) -> FBSDENet:
    net.to(DEVICE)
    net.train()

    opt = optim.Adam(net.parameters(), lr=cfg["fbsde_lr"])
    scheduler = optim.lr_scheduler.StepLR(
        opt,
        step_size=cfg["fbsde_decay_step"],
        gamma=0.1,
    )
    mse = nn.MSELoss()

    N, D = cfg["N_steps"], delay_steps(cfg)
    batch = cfg["batch_size"]

    for it in range(cfg["fbsde_iters"]):
        dB = torch.randn(batch, N, device=DEVICE) * math.sqrt(dt(cfg))

        x = torch.full((batch,), cfg["x0"], device=DEVICE)
        v_hist = torch.zeros(batch, N + 1, device=DEVICE)
        h, c = net.init_hidden(batch, DEVICE)

        Y_pred = [None] * (N + 1)
        Z_pred = [None] * (N + 1)
        EY_pred = [None] * (N + 1)
        tildeY = [None] * (N + 1)

        t0 = torch.zeros(batch, device=DEVICE)
        y, z, ey, h, c = net.forward_step(t0, x, h, c)
        Y_pred[0], Z_pred[0], EY_pred[0] = y, z, ey

        for i in range(N):
            y_i, z_i, ey_i = Y_pred[i], Z_pred[i], EY_pred[i]
            ey_used = ey_i if i <= N - D else torch.zeros_like(ey_i)

            v_val = - (cfg["b0"] * y_i + cfg["b1"] * ey_used) / (2.0 * cfg["R"])
            v = torch.relu(v_val)
            v_hist[:, i] = v

            v_del = get_v_delayed(v_hist, i, D, DEVICE)
            x = x + drift_x(x, v, v_del, cfg) * dt(cfg) + cfg["sigma"] * dB[:, i]

            tildeY[i + 1] = y_i - cfg["a0"] * y_i * dt(cfg) + z_i * dB[:, i]

            t_next = torch.full((batch,), ((i + 1) * dt(cfg)) / cfg["T"], device=DEVICE)
            y, z, ey, h, c = net.forward_step(t_next, x, h, c)
            Y_pred[i + 1], Z_pred[i + 1], EY_pred[i + 1] = y, z, ey

        L1 = 0.0
        for i in range(N):
            L1 = L1 + mse(Y_pred[i + 1], tildeY[i + 1])
        L1 = L1 + mse(Y_pred[N], torch.full((batch,), cfg["S0"], device=DEVICE))

        L2 = 0.0
        for i in range(0, N - D + 1):
            L2 = L2 + mse(EY_pred[i], tildeY[i + D])

        L1 = L1 / max(1, N)
        L2 = L2 / max(1, N - D + 1)
        loss = L1 + L2

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        scheduler.step()

        if (it + 1) % 500 == 0:
            print(
                f"[Alg2 MV-FABSDE] iter={it+1}, "
                f"L1={L1.item():.6f}, L2={L2.item():.6f}, total={loss.item():.6f}"
            )

    return net


def rollout_fbsde_deterministic(net: FBSDENet, cfg: Dict) -> np.ndarray:
    net.eval()

    N, D = cfg["N_steps"], delay_steps(cfg)

    x = torch.full((1,), cfg["x0"], device=DEVICE)
    v_hist = torch.zeros(1, N + 1, device=DEVICE)
    h, c = net.init_hidden(1, DEVICE)

    vs = []

    with torch.no_grad():
        for i in range(N):
            t_norm = torch.full((1,), (i * dt(cfg)) / cfg["T"], device=DEVICE)
            y, z, ey, h, c = net.forward_step(t_norm, x, h, c)

            ey_used = ey if i <= N - D else torch.zeros_like(ey)
            v_val = - (cfg["b0"] * y + cfg["b1"] * ey_used) / (2.0 * cfg["R"])
            v = torch.relu(v_val)

            vs.append(float(v.item()))

            v_hist[:, i] = v
            v_del = get_v_delayed(v_hist, i, D, DEVICE)
            x = x + drift_x(x, v, v_del, cfg) * dt(cfg)

    return np.array(vs, dtype=np.float64)


# ============================================================
# 6. Benchmark
# ============================================================
def analytic_solution(cfg: Dict) -> Tuple[np.ndarray, np.ndarray]:
    T, N = cfg["T"], cfg["N_steps"]
    a0, b0, b1 = cfg["a0"], cfg["b0"], cfg["b1"]
    R, S0 = cfg["R"], cfg["S0"]
    D = delay_steps(cfg)

    t_grid = np.linspace(0, T, N + 1)
    y = S0 * np.exp(a0 * (T - t_grid))

    u = np.zeros_like(y)
    for i in range(N + 1):
        term1 = b0 * y[i]
        term2 = b1 * y[i + D] if (i + D) <= N else 0.0
        u[i] = max(0.0, - (term1 + term2) / (2.0 * R))

    return t_grid, u


def mae_rmse(y_hat: np.ndarray, y_ref: np.ndarray) -> Tuple[float, float]:
    y_hat = np.asarray(y_hat).reshape(-1)
    y_ref = np.asarray(y_ref).reshape(-1)
    err = y_hat - y_ref
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    return mae, rmse


# ============================================================
# 7. P-PGDPO Projection
# ============================================================
def estimate_costate_mc(
    policy: LSTMPolicy,
    cfg: Dict,
    n0: int,
    x0_val: torch.Tensor,
    h0: torch.Tensor,
    c0: torch.Tensor,
    v_hist0: torch.Tensor,
    M: int,
) -> torch.Tensor:
    policy.eval()

    N = int(cfg["N_steps"])
    D = delay_steps(cfg)
    M = int(M)

    # ------------------------------------------------------------
    # 1) Normalize initial batch dimension B
    # ------------------------------------------------------------
    x0_base = x0_val.detach().reshape(-1)
    B = int(x0_base.numel())

    h0_base = h0.detach()
    c0_base = c0.detach()
    v_hist_base = v_hist0.detach()

    if h0_base.dim() == 1:
        h0_base = h0_base.unsqueeze(0)
    if c0_base.dim() == 1:
        c0_base = c0_base.unsqueeze(0)
    if v_hist_base.dim() == 1:
        v_hist_base = v_hist_base.unsqueeze(0)

    if h0_base.size(0) == 1 and B > 1:
        h0_base = h0_base.repeat(B, 1)
    if c0_base.size(0) == 1 and B > 1:
        c0_base = c0_base.repeat(B, 1)
    if v_hist_base.size(0) == 1 and B > 1:
        v_hist_base = v_hist_base.repeat(B, 1)

    if h0_base.size(0) != B:
        raise ValueError(f"h0 batch mismatch: h0 has {h0_base.size(0)}, x0 has {B}")
    if c0_base.size(0) != B:
        raise ValueError(f"c0 batch mismatch: c0 has {c0_base.size(0)}, x0 has {B}")
    if v_hist_base.size(0) != B:
        raise ValueError(f"v_hist0 batch mismatch: v_hist0 has {v_hist_base.size(0)}, x0 has {B}")

    # ------------------------------------------------------------
    # 2) Expand B initial states into B x M Monte Carlo paths
    # ------------------------------------------------------------
    x_init = x0_base.clone().detach().requires_grad_(True)

    BM = B * M

    x = x_init[:, None].expand(B, M).reshape(BM)

    h = (
        h0_base[:, None, :]
        .expand(B, M, h0_base.size(1))
        .reshape(BM, h0_base.size(1))
        .contiguous()
    )
    c = (
        c0_base[:, None, :]
        .expand(B, M, c0_base.size(1))
        .reshape(BM, c0_base.size(1))
        .contiguous()
    )

    v_hist = (
        v_hist_base[:, None, :]
        .expand(B, M, v_hist_base.size(1))
        .reshape(BM, v_hist_base.size(1))
        .clone()
        .contiguous()
    )

    cost = torch.zeros(BM, device=DEVICE, dtype=x.dtype)

    # ------------------------------------------------------------
    # 3) Vectorized rollout from n0 to N
    # ------------------------------------------------------------
    for n in range(n0, N):
        t_norm = torch.full(
            (BM,),
            (n * dt(cfg)) / float(cfg["T"]),
            device=DEVICE,
            dtype=x.dtype,
        )

        v, h, c = policy.forward_step(t_norm, x, h, c)

        v_hist[:, n] = v
        v_del = get_v_delayed(v_hist, n, D, DEVICE)

        dB = math.sqrt(dt(cfg)) * torch.randn(BM, device=DEVICE, dtype=x.dtype)

        x = (
            x
            + drift_x(x, v, v_del, cfg) * dt(cfg)
            + float(cfg["sigma"]) * dB
        )

        cost = cost + float(cfg["R"]) * v**2 * dt(cfg)

    cost = cost + float(cfg["S0"]) * x

    # ------------------------------------------------------------
    # 4) Mean over Monte Carlo paths, one gradient per initial state
    # ------------------------------------------------------------
    cost_by_state = cost.view(B, M)
    J_by_state = cost_by_state.mean(dim=1)

    grad_x0, = torch.autograd.grad(
        J_by_state.sum(),
        x_init,
        retain_graph=False,
        create_graph=False,
    )

    return grad_x0.detach()


def project_p_pgdpo(
    policy_pg: LSTMPolicy,
    cfg: Dict,
    t_grid: np.ndarray,
    rollout_pg: Dict[str, List[torch.Tensor]],
) -> Tuple[np.ndarray, np.ndarray, Dict]:
    print(">>> Computing P-PGDPO projection...")

    N = int(cfg["N_steps"])
    D = delay_steps(cfg)
    M = int(cfg["N_mc_stage2"])
    B_branch = int(cfg["N_branch"])

    idx_list = sorted(
        set(
            int(round(i * (N - 1) / (cfg["N_compare"] - 1)))
            for i in range(cfg["N_compare"])
        )
    )

    xs = rollout_pg["xs"]
    hs = rollout_pg["hs"]
    cs = rollout_pg["cs"]
    v_hists = rollout_pg["v_hists"]

    v_pmp_list = []
    t_cmp = []
    step_times = []

    total_t0 = time.time()

    for n in idx_list:
        step_t0 = time.time()

        # --------------------------------------------------------
        # 1) Current costate lambda_t
        # --------------------------------------------------------
        lam_t = estimate_costate_mc(
            policy_pg,
            cfg,
            n,
            xs[n],
            hs[n],
            cs[n],
            v_hists[n],
            M=M,
        ).reshape(-1)[0].item()

        # --------------------------------------------------------
        # 2) Anticipated future costate lambda_{t+delta}
        # --------------------------------------------------------
        if n + D >= N:
            lam_t_delta = float(cfg["S0"]) if (n + D) == N else 0.0

        else:
            # ----------------------------------------------------
            # 2-a) Vectorized branch simulation from n to n+D
            # ----------------------------------------------------
            x_curr = xs[n].detach().reshape(1).repeat(B_branch)

            h_curr = hs[n].detach()
            c_curr = cs[n].detach()
            v_hist_curr = v_hists[n].detach()

            if h_curr.dim() == 1:
                h_curr = h_curr.unsqueeze(0)
            if c_curr.dim() == 1:
                c_curr = c_curr.unsqueeze(0)
            if v_hist_curr.dim() == 1:
                v_hist_curr = v_hist_curr.unsqueeze(0)

            h_curr = h_curr.repeat(B_branch, 1)
            c_curr = c_curr.repeat(B_branch, 1)
            v_hist_curr = v_hist_curr.repeat(B_branch, 1).clone()

            with torch.no_grad():
                for k in range(n, n + D):
                    t_k_norm = torch.full(
                        (B_branch,),
                        (k * dt(cfg)) / float(cfg["T"]),
                        device=DEVICE,
                        dtype=x_curr.dtype,
                    )

                    v, h_curr, c_curr = policy_pg.forward_step(
                        t_k_norm,
                        x_curr,
                        h_curr,
                        c_curr,
                    )

                    v_hist_curr[:, k] = v
                    v_del = get_v_delayed(v_hist_curr, k, D, DEVICE)

                    dB = math.sqrt(dt(cfg)) * torch.randn(
                        B_branch,
                        device=DEVICE,
                        dtype=x_curr.dtype,
                    )

                    x_curr = (
                        x_curr
                        + drift_x(x_curr, v, v_del, cfg) * dt(cfg)
                        + float(cfg["sigma"]) * dB
                    )

            # ----------------------------------------------------
            # 2-b) Future costate for all branches at once
            # ----------------------------------------------------
            lam_future = estimate_costate_mc(
                policy_pg,
                cfg,
                n + D,
                x_curr,
                h_curr,
                c_curr,
                v_hist_curr,
                M=M,
            )

            lam_t_delta = float(lam_future.mean().item())

        # --------------------------------------------------------
        # 3) Closed-form projection
        # --------------------------------------------------------
        val = -(
            float(cfg["b0"]) * lam_t
            + float(cfg["b1"]) * lam_t_delta
        ) / (2.0 * float(cfg["R"]))

        v_pmp_list.append(max(0.0, val))
        t_cmp.append(t_grid[n])

        step_times.append(time.time() - step_t0)

    total_elapsed = time.time() - total_t0
    step_times = np.array(step_times, dtype=np.float64)

    timing = {
        "pgdpo_projection_total_sec": float(total_elapsed),
        "pgdpo_projection_avg_step_sec": float(np.mean(step_times)) if len(step_times) > 0 else np.nan,
        "pgdpo_projection_min_step_sec": float(np.min(step_times)) if len(step_times) > 0 else np.nan,
        "pgdpo_projection_max_step_sec": float(np.max(step_times)) if len(step_times) > 0 else np.nan,
        "pgdpo_projection_step_times": step_times,
        "pgdpo_projection_n_points": int(len(step_times)),
    }

    return (
        np.array(t_cmp, dtype=np.float64),
        np.array(v_pmp_list, dtype=np.float64),
        timing,
    )


# ============================================================
# 8. Metrics + Single Seed Driver
# ============================================================
def compute_metrics_and_curves(
    policy_pg: LSTMPolicy,
    fbsde_net: FBSDENet,
    cfg: Dict,
) -> Dict:
    policy_pg.eval()
    fbsde_net.eval()

    t_grid_full, v_star_full = analytic_solution(cfg)
    t_plot = t_grid_full[:-1]
    v_star = v_star_full[:-1]

    rollout_pg = rollout_policy_pg_deterministic(policy_pg, cfg)
    v_pg = np.array([v.item() for v in rollout_pg["vs"]], dtype=np.float64)

    v_fbsde = rollout_fbsde_deterministic(fbsde_net, cfg)

    t_cmp, v_pmp, proj_timing = project_p_pgdpo(
        policy_pg,
        cfg,
        t_grid_full,
        rollout_pg,
    )

    v_star_cmp = np.interp(t_cmp, t_grid_full, v_star_full)
    v_pg_cmp = np.interp(t_cmp, t_plot, v_pg)
    v_fbsde_cmp = np.interp(t_cmp, t_plot, v_fbsde)

    mae_pg, rmse_pg = mae_rmse(v_pg_cmp, v_star_cmp)
    mae_fbsde, rmse_fbsde = mae_rmse(v_fbsde_cmp, v_star_cmp)
    mae_pmp, rmse_pmp = mae_rmse(v_pmp, v_star_cmp)

    v_pmp_on_N = np.interp(t_plot, t_cmp, v_pmp)

    return {
        "t_plot": t_plot,
        "v_star": v_star,

        "v_pg": v_pg,
        "v_fbsde": v_fbsde,
        "v_pmp": v_pmp_on_N,

        "t_cmp": t_cmp,
        "v_pmp_cmp": v_pmp,

        "pg_rmse": rmse_pg,
        "pg_mae": mae_pg,

        "fbsde_rmse": rmse_fbsde,
        "fbsde_mae": mae_fbsde,

        "pgdpo_rmse": rmse_pmp,
        "pgdpo_mae": mae_pmp,

        "pgdpo_projection_total_sec": proj_timing["pgdpo_projection_total_sec"],
        "pgdpo_projection_avg_step_sec": proj_timing["pgdpo_projection_avg_step_sec"],
        "pgdpo_projection_min_step_sec": proj_timing["pgdpo_projection_min_step_sec"],
        "pgdpo_projection_max_step_sec": proj_timing["pgdpo_projection_max_step_sec"],
        "pgdpo_projection_step_times": proj_timing["pgdpo_projection_step_times"],
        "pgdpo_projection_n_points": proj_timing["pgdpo_projection_n_points"],
    }


def run_single_seed(seed: int, cfg_base: Dict) -> Dict:
    cfg = cfg_for_seed(cfg_base, seed)
    set_seed(seed)

    print("\n" + "=" * 100)
    print(
        f"[RUN] seed={seed} | T={cfg['T']}, N={cfg['N_steps']}, "
        f"dt={dt(cfg):.4f}, delta={cfg['delta']}, D={delay_steps(cfg)}, device={DEVICE}"
    )
    print("=" * 100)

    seed_t0 = time.time()

    # ------------------------------------------------------------
    # 1) Warm-up / LSTM-DPO training timing
    # ------------------------------------------------------------
    warmup_t0 = time.time()
    policy_pg = LSTMPolicy(hidden_size=cfg["hidden"])
    policy_pg = warmup_train(policy_pg, cfg)
    warmup_elapsed_sec = float(time.time() - warmup_t0)

    # ------------------------------------------------------------
    # 2) MV-FABSDE training timing
    # ------------------------------------------------------------
    fbsde_t0 = time.time()
    fbsde_net = FBSDENet(hidden_size=cfg["hidden"])
    fbsde_net = train_fbsde(fbsde_net, cfg)
    fbsde_elapsed_sec = float(time.time() - fbsde_t0)

    # ------------------------------------------------------------
    # 3) Metrics + PGDPO projection timing
    # ------------------------------------------------------------
    eval_t0 = time.time()
    out = compute_metrics_and_curves(policy_pg, fbsde_net, cfg)
    eval_elapsed_sec = float(time.time() - eval_t0)

    single_seed_total_sec = float(time.time() - seed_t0)

    out["seed"] = seed

    # Backward-compatible name
    out["elapsed_sec"] = single_seed_total_sec

    # Explicit timing keys
    out["warmup_train_sec"] = warmup_elapsed_sec
    out["fbsde_train_sec"] = fbsde_elapsed_sec
    out["metrics_and_projection_sec"] = eval_elapsed_sec
    out["single_seed_total_sec"] = single_seed_total_sec

    print("\n[Seed Metrics]")
    print(f"RMSE & MAE (PG)      : {out['pg_rmse']:.6f}, {out['pg_mae']:.6f}")
    print(f"RMSE & MAE (FBSDE)   : {out['fbsde_rmse']:.6f}, {out['fbsde_mae']:.6f}")
    print(f"RMSE & MAE (P-PGDPO) : {out['pgdpo_rmse']:.6f}, {out['pgdpo_mae']:.6f}")

    print("\n[Training / Seed Timing]")
    print(f"Warmup training time              : {out['warmup_train_sec']:.6f} sec")
    print(f"MV-FABSDE training time           : {out['fbsde_train_sec']:.6f} sec")
    print(f"Metrics + projection eval time    : {out['metrics_and_projection_sec']:.6f} sec")
    print(f"Single-seed total experiment time : {out['single_seed_total_sec']:.6f} sec")

    print("\n[Projection Timing]")
    print(
        f"P-PGDPO projection points        : "
        f"{out['pgdpo_projection_n_points']} "
        f"(requested N_compare={cfg['N_compare']}, N_steps={cfg['N_steps']})"
    )
    print(f"P-PGDPO projection total time    : {out['pgdpo_projection_total_sec']:.6f} sec")
    print(f"P-PGDPO projection avg/point     : {out['pgdpo_projection_avg_step_sec']:.6f} sec")
    print(f"P-PGDPO projection min/point     : {out['pgdpo_projection_min_step_sec']:.6f} sec")
    print(f"P-PGDPO projection max/point     : {out['pgdpo_projection_max_step_sec']:.6f} sec")

    print(f"\n[Elapsed] {out['elapsed_sec']:.2f} sec")

    del policy_pg, fbsde_net
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


# ============================================================
# 9. Multi-seed Summary + Band Plotting
# ============================================================
def metric_mean_std(results: List[Dict], key: str) -> Tuple[float, float]:
    vals = np.array([r[key] for r in results], dtype=np.float64)
    mean = float(np.mean(vals))
    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    return mean, std


def print_seedwise_table(results: List[Dict]) -> None:
    print("\n" + "#" * 100)
    print("Seed-wise Metrics")
    print("#" * 100)

    header = f"{'seed':>5} | {'Algorithm':<10} | {'RMSE':>12} | {'MAE':>12}"
    print(header)
    print("-" * len(header))

    algos = [
        ("PG", "pg"),
        ("FBSDE", "fbsde"),
        ("P-PGDPO", "pgdpo"),
    ]

    for r in results:
        for name, key in algos:
            print(
                f"{r['seed']:5d} | {name:<10} | "
                f"{r[f'{key}_rmse']:12.6e} | {r[f'{key}_mae']:12.6e}"
            )


def print_multiseed_summary(results: List[Dict]) -> None:
    print("\n" + "#" * 100)
    print("Multi-seed Summary: mean +/- std over seeds")
    print("#" * 100)

    header = (
        f"{'Algorithm':<10} | "
        f"{'RMSE mean':>12} | {'RMSE std':>12} | "
        f"{'MAE mean':>12} | {'MAE std':>12}"
    )
    print(header)
    print("-" * len(header))

    algos = [
        ("PG", "pg"),
        ("FBSDE", "fbsde"),
        ("P-PGDPO", "pgdpo"),
    ]

    for name, key in algos:
        rmse_mean, rmse_std = metric_mean_std(results, f"{key}_rmse")
        mae_mean, mae_std = metric_mean_std(results, f"{key}_mae")

        print(
            f"{name:<10} | "
            f"{rmse_mean:12.6e} | {rmse_std:12.6e} | "
            f"{mae_mean:12.6e} | {mae_std:12.6e}"
        )


def print_projection_timing_summary(results: List[Dict]) -> None:
    print("\n" + "#" * 100)
    print("Projection Timing Summary: mean +/- std over seeds")
    print("#" * 100)

    rows = [
        ("Total projection sec", "pgdpo_projection_total_sec"),
        ("Avg sec / projected point", "pgdpo_projection_avg_step_sec"),
        ("Min sec / projected point", "pgdpo_projection_min_step_sec"),
        ("Max sec / projected point", "pgdpo_projection_max_step_sec"),
    ]

    header = f"{'Metric':<30} | {'mean':>12} | {'std':>12}"
    print(header)
    print("-" * len(header))

    for name, key in rows:
        vals = np.array([r[key] for r in results], dtype=np.float64)
        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0

        print(f"{name:<30} | {mean:12.6e} | {std:12.6e}")


def print_training_timing_summary(results: List[Dict]) -> None:
    print("\n" + "#" * 100)
    print("Training / Single-seed Timing Summary: mean +/- std over seeds")
    print("#" * 100)

    rows = [
        ("Warmup training sec", "warmup_train_sec"),
        ("MV-FABSDE training sec", "fbsde_train_sec"),
        ("Metrics + projection sec", "metrics_and_projection_sec"),
        ("Single-seed total sec", "single_seed_total_sec"),
    ]

    header = f"{'Metric':<32} | {'mean':>12} | {'std':>12}"
    print(header)
    print("-" * len(header))

    for name, key in rows:
        vals = np.array([r[key] for r in results], dtype=np.float64)
        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0

        print(f"{name:<32} | {mean:12.6e} | {std:12.6e}")


def curve_mean_std(curves: List[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    arr = np.stack(curves, axis=0).astype(np.float64)
    mean = np.mean(arr, axis=0)
    std = np.std(arr, axis=0, ddof=1) if arr.shape[0] > 1 else np.zeros_like(mean)
    return mean, std


def plot_band_figure(
    t_grid: np.ndarray,
    gt: np.ndarray,
    curves_by_algo: Dict[str, List[np.ndarray]],
    save_path: str,
) -> None:
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(t_grid, gt, "k-", lw=2.2, label="Benchmark")

    style = {
        "PG": {
            "color": "tab:blue",
            "mean_ls": ":",
            "label": "LSTM-DPO",
        },
        "FBSDE": {
            "color": "tab:green",
            "mean_ls": "-.",
            "label": "DEEP ABSDE",
        },
        "P-PGDPO": {
            "color": "tab:red",
            "mean_ls": "--",
            "label": "PGDPO",
        },
    }

    for algo in ["PG", "FBSDE", "P-PGDPO"]:
        mean_curve, std_curve = curve_mean_std(curves_by_algo[algo])
        upper = mean_curve + std_curve
        lower = mean_curve - std_curve
        color = style[algo]["color"]

        ax.plot(
            t_grid,
            mean_curve,
            linestyle=style[algo]["mean_ls"],
            color=color,
            lw=2.0,
            label=style[algo]["label"],
        )

        ax.fill_between(
            t_grid,
            lower,
            upper,
            color=color,
            alpha=0.16,
            linewidth=0.0,
        )

        # Band upper/lower boundaries as dotted curves.
        ax.plot(t_grid, upper, linestyle=":", color=color, lw=1.25, alpha=0.95)
        ax.plot(t_grid, lower, linestyle=":", color=color, lw=1.25, alpha=0.95)

    ax.set_xlabel("Time")
    ax.set_ylabel("Advertising Expenditure")
    ax.legend()
    ax.grid(alpha=0.1)
    fig.tight_layout()

    #fig.savefig(save_path, format="pdf", bbox_inches="tight")
    #print(f"[Saved] {save_path}")

    plt.show()
    plt.close(fig)


def run_multiseed_experiment(cfg: Dict) -> Dict:
    os.makedirs(cfg["outdir"], exist_ok=True)

    results = []
    for seed in cfg["seeds"]:
        results.append(run_single_seed(seed, cfg))

    print_seedwise_table(results)
    print_multiseed_summary(results)
    print_training_timing_summary(results)
    print_projection_timing_summary(results)

    ref = results[0]
    t_grid = ref["t_plot"]
    gt = ref["v_star"]

    curves_by_algo = {
        "PG": [r["v_pg"] for r in results],
        "FBSDE": [r["v_fbsde"] for r in results],
        "P-PGDPO": [r["v_pmp"] for r in results],
    }

    fig_path = os.path.join(
        cfg["outdir"],
        f"benchmark2_control_multiseed_band_seeds{cfg['seeds'][0]}to{cfg['seeds'][-1]}.pdf",
    )

    plot_band_figure(
        t_grid=t_grid,
        gt=gt,
        curves_by_algo=curves_by_algo,
        save_path=fig_path,
    )
    '''
    zip_path = os.path.join(cfg["outdir"], "benchmark2_multiseed_figures.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for fname in os.listdir(cfg["outdir"]):
            if fname.endswith(".pdf"):
                full = os.path.join(cfg["outdir"], fname)
                zipf.write(full, arcname=fname)

    print(f"[ZIP saved] {zip_path}")
    '''
    return {
        "cfg": cfg,
        "results": results,
        "fig_path": fig_path,
        #"zip_path": zip_path,
    }


# ============================================================
# 10. Execute
# ============================================================
RESULT = run_multiseed_experiment(CFG)
